# Khmer AI-Text Detector — Notebook 1: Collect human-written Khmer text

**Goal:** build `human.csv` — Khmer passages written by people (label = 0).

**Sources**
1. **Khmer Wikipedia** (CC BY-SA 4.0) — reliable, loads from Hugging Face.
2. **Khmer web news from mC4** (ODC-BY, Common Crawl) — mostly Cambodian news sites, crawled 2013–2020, so written before ChatGPT. Loads from Hugging Face.
3. **VOA Khmer** news (generally US public domain) — scraped via the site's sitemaps, keeping only articles published **before 30 Nov 2022** (ChatGPT release), so they are almost surely human-written. **Optional:** if the site can't be reached (it currently blocks scrapers), these cells are skipped and the notebook still finishes.

**How to run:** `Runtime → Run all`. No GPU needed. Takes ~20–40 min (VOA scraping is slow on purpose, to be polite to the server).
If a cell prints an error or `0 articles`, copy the output and send it to Claude.

In [1]:
%pip -q install datasets beautifulsoup4 lxml requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# In Colab: save to Google Drive so it survives restarts. Locally: save to the project's ./data folder.
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/khmer-ai-detector'
else:
    # notebooks run with cwd = notebooks/, so step up to the project root
    PROJECT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

DATA = f'{PROJECT}/data'
os.makedirs(DATA, exist_ok=True)
print('Saving to', DATA)

Saving to C:\Users\Asus\Documents\claude\khmer-ai-text-detector/data


## Settings

In [3]:
WIKI_MAX     = 4000     # max Wikipedia passages to keep
WEB_MAX      = 4000     # max mC4 web-news passages to keep
WEB_SHARDS   = 1        # mC4 Khmer files to download (~19 MB each, ~10k usable passages each)
VOA_MAX      = 3000     # max VOA articles to scrape
CUTOFF_DATE  = '2022-11-30'   # keep only VOA articles published before this
MIN_CHARS, MAX_CHARS = 300, 2500   # passage length range
MIN_KHMER_RATIO = 0.6   # share of letters that must be Khmer script
SLEEP_SEC    = 1.0      # pause between VOA requests
SEED = 42

## Helper functions (cleaning)

In [4]:
import re, unicodedata, hashlib

KHMER_RE = re.compile(r'[\u1780-\u17FF\u19E0-\u19FF]')

def normalize(text: str) -> str:
    text = unicodedata.normalize('NFC', text)
    text = text.replace('\u200b', '')          # zero-width space (common in Khmer)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    return text.strip()

def khmer_ratio(text: str) -> float:
    letters = [c for c in text if not c.isspace() and not c.isdigit() and c not in '.,;:!?()[]"\'-–—/«»។៕']
    if not letters:
        return 0.0
    return sum(bool(KHMER_RE.match(c)) for c in letters) / len(letters)

def make_passage(text: str, max_chars: int = MAX_CHARS) -> str:
    """Take whole paragraphs from the start until max_chars is reached."""
    out = ''
    for para in text.split('\n'):
        para = para.strip()
        if not para:
            continue
        if len(out) + len(para) + 1 > max_chars:
            break
        out = (out + '\n' + para) if out else para
    return out

def is_good(text: str) -> bool:
    return MIN_CHARS <= len(text) <= MAX_CHARS and khmer_ratio(text) >= MIN_KHMER_RATIO

def text_id(text: str) -> str:
    return hashlib.md5(text.encode('utf-8')).hexdigest()[:12]

print(khmer_ratio('ប្រទេសកម្ពុជា គឺជាប្រទេសមួយ'), khmer_ratio('hello world'))

1.0 0.0


## Source 1 — Khmer Wikipedia

In [5]:
from datasets import load_dataset
import pandas as pd

wiki = load_dataset('wikimedia/wikipedia', '20231101.km', split='train')
print(wiki)

# Drop reference/link sections that often appear at the end of articles
CUT_HEADINGS = ['ឯកសារយោង', 'តំណភ្ជាប់', 'សូមមើលផងដែរ', 'References', 'External links']

def clean_wiki(text):
    for h in CUT_HEADINGS:
        i = text.find(h)
        if i > 0:
            text = text[:i]
    return normalize(text)

rows = []
for ex in wiki.shuffle(seed=SEED):
    p = make_passage(clean_wiki(ex['text']))
    if is_good(p):
        rows.append({'text': p, 'title': ex['title'], 'source': 'wikipedia',
                     'url': ex['url'], 'date': '', 'license': 'CC BY-SA 4.0'})
    if len(rows) >= WIKI_MAX:
        break

wiki_df = pd.DataFrame(rows)
wiki_df.to_csv(f'{DATA}/raw_wikipedia.csv', index=False)
print(len(wiki_df), 'Wikipedia passages')
wiki_df.head(3)

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 11994
})


4000 Wikipedia passages


,text,title,source,url,date,license
0,() ស្ថិតនៅក្នុង ស្រុកកំពង់រោទិ៍ ខេត្តស្វាយរៀង ...,ឃុំរាជមន្ទីរ,wikipedia,https://km.wikipedia.org/wiki/%E1%9E%83%E1%9E%...,,CC BY-SA 4.0
1,ខេត្តផុងសាលី (ជាភាសាឡាវ៖ ຜົງສາລີ) (ឈ្មោះជាខ្មែ...,ខេត្តផុងសាលី,wikipedia,https://km.wikipedia.org/wiki/%E1%9E%81%E1%9F%...,,CC BY-SA 4.0
2,សាសនាហិណ្ឌូ (;;) គឺជាសាសនាដ៏ចំណាស់មួយ ដែលចាប់ក...,ហិណ្ឌូសាសនា,wikipedia,https://km.wikipedia.org/wiki/%E1%9E%A0%E1%9E%...,,CC BY-SA 4.0


## Source 2 — Khmer web news (mC4, crawled 2013–2020)
The Khmer part of [mC4](https://huggingface.co/datasets/allenai/c4) — Common Crawl pages, mostly Cambodian news sites. Short lines (menus, bylines, ads) are dropped; only long, mostly-Khmer lines are kept.

In [6]:
MC4_BASE = 'https://huggingface.co/datasets/allenai/c4/resolve/main/multilingual/'
web = load_dataset('json', split='train',
                   data_files=[f'{MC4_BASE}c4-km.tfrecord-{i:05d}-of-00064.json.gz' for i in range(WEB_SHARDS)])
print(web)

def clean_web(text):
    lines = [l.strip() for l in normalize(text).split('\n')]
    return '\n'.join(l for l in lines if len(l) >= 80 and khmer_ratio(l) >= 0.8)

rows = []
for ex in web.shuffle(seed=SEED):
    p = make_passage(clean_web(ex['text']))
    if is_good(p):
        rows.append({'text': p, 'title': '', 'source': 'mc4_web',
                     'url': ex['url'], 'date': str(ex['timestamp'])[:10], 'license': 'ODC-BY (mC4)'})
    if len(rows) >= WEB_MAX:
        break

web_df = pd.DataFrame(rows)
web_df.to_csv(f'{DATA}/raw_mc4_web.csv', index=False)
print(len(web_df), 'web passages')
print(web_df['url'].str.split('/').str[2].value_counts().head(10))
web_df.head(3)

Dataset({
    features: ['text', 'timestamp', 'url'],
    num_rows: 11822
})


4000 web passages
url
www.khtoem.com               385
www.rasmeinews.com           175
khmer.voanews.com            147
postnews.com.kh               92
www.cen.com.kh                89
kohsantepheapdaily.com.kh     85
nokorwatnews.com              68
www.khmerload.com             52
khmeread.com                  51
kampucheathmey.com            50
Name: count, dtype: int64


,text,title,source,url,date,license
0,លោកជំទាវកិត្តិសង្គហបណ្ឌិត ម៉ែន សំអន បានអញ្ជើញច...,,mc4_web,http://monasri.gov.kh/archives/3806,2019-11-21,ODC-BY (mC4)
1,នៅក្នុងខែគុម្ភៈខណៈដែលការលក់ បានធ្លាក់ចុះតាមនោះ...,,mc4_web,http://www.khmeronlinenews.com/%E1%9E%A0%E1%9E...,2017-07-25,ODC-BY (mC4)
2,“ជាដំបូង… គឺខ្ញុំនឹងទាក់ទងខាងរដ្ឋបាលព្រៃឈើ ដើម...,,mc4_web,https://sroksrear.wordpress.com/2012/11/21/chi...,2018-03-19,ODC-BY (mC4)


## Source 3 — VOA Khmer (news before Nov 2022, optional)
First a quick **test cell**: it checks the site is reachable and that article parsing works. Run it and look at the output before the long scrape.

In [7]:
import requests, time, random
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE = 'https://www.voakhmer.com'
S = requests.Session()
S.headers['User-Agent'] = 'Mozilla/5.0 (student research project: Khmer NLP dataset)'

def get(url, timeout=30):
    r = S.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text

def sitemaps_from_robots():
    txt = get(f'{BASE}/robots.txt')
    maps = [l.split(':', 1)[1].strip() for l in txt.splitlines() if l.lower().startswith('sitemap:')]
    return maps or [f'{BASE}/sitemap.xml']

def parse_sitemap(xml):
    """Returns (child_sitemaps, [(url, lastmod), ...])."""
    soup = BeautifulSoup(xml, 'xml')
    children = [l.text.strip() for l in soup.select('sitemap > loc')]
    urls = []
    for u in soup.find_all('url'):
        loc = u.find('loc')
        lm = u.find('lastmod')
        if loc:
            urls.append((loc.text.strip(), lm.text.strip()[:10] if lm else ''))
    return children, urls

def parse_article(html):
    soup = BeautifulSoup(html, 'lxml')
    meta = lambda **kw: (soup.find('meta', attrs=kw) or {}).get('content', '')
    title = meta(property='og:title') or (soup.title.text if soup.title else '')
    date = meta(property='article:published_time') or meta(name='pubdate')
    if not date:
        t = soup.find('time', attrs={'datetime': True})
        date = t['datetime'] if t else ''
    body = soup.select_one('div.wsw') or soup.select_one('#article-content') or soup.find('article') or soup
    paras = [p.get_text(' ', strip=True) for p in body.find_all('p')]
    text = normalize('\n'.join(p for p in paras if len(p) > 20))
    return {'title': normalize(title), 'date': date[:10], 'text': text}

# ---- TEST ---- (VOA is optional: if the site is unreachable, the VOA cells are skipped)
try:
    maps = sitemaps_from_robots()
    print('Sitemaps found:', maps[:5])
    children, urls = parse_sitemap(get(maps[0]))
    print('Child sitemaps:', len(children), children[:3])
    print('URLs in first sitemap:', len(urls), urls[:3])
    home = BeautifulSoup(get(BASE), 'lxml')
    links = [urljoin(BASE, a['href']) for a in home.find_all('a', href=True) if '/a/' in a['href']]
    print('Article links on homepage:', len(links))
    if links:
        art = parse_article(get(links[0]))
        print('\nTEST ARTICLE:', links[0]); print('title:', art['title']); print('date:', art['date'])
        print('chars:', len(art['text']), '| khmer ratio: %.2f' % khmer_ratio(art['text']))
        print(art['text'][:300])
    VOA_OK = True
except Exception as e:
    VOA_OK = False
    print('VOA Khmer is not reachable, skipping VOA:', e)


VOA Khmer is not reachable, skipping VOA: HTTPSConnectionPool(host='www.voakhmer.com', port=443): Max retries exceeded with url: /robots.txt (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1081)')))


### Collect old article URLs from the sitemaps

In [8]:
from tqdm.auto import tqdm

def collect_urls(max_sitemaps=400):
    queue, seen, found = list(sitemaps_from_robots()), set(), {}
    pbar = tqdm(desc='sitemaps')
    while queue and len(seen) < max_sitemaps:
        sm = queue.pop(0)
        if sm in seen:
            continue
        seen.add(sm); pbar.update(1)
        try:
            children, urls = parse_sitemap(get(sm))
        except Exception as e:
            print('skip', sm, e); continue
        queue.extend(children)
        for u, lm in urls:
            if '/a/' in u:
                found[u] = lm
        time.sleep(0.3)
    pbar.close()
    return found

found = collect_urls() if VOA_OK else {}
print('Article URLs found:', len(found))
# Prefer URLs whose lastmod is before the cutoff (if the sitemap gives dates)
old = [u for u, lm in found.items() if lm and lm < CUTOFF_DATE]
nodate = [u for u, lm in found.items() if not lm]
print('Before cutoff:', len(old), '| no date:', len(nodate))
candidates = old + nodate
random.Random(SEED).shuffle(candidates)

Article URLs found: 0
Before cutoff: 0 | no date: 0


### Scrape the articles (slow — resumes automatically if interrupted)

In [9]:
import json
CACHE = f'{DATA}/voa_cache.jsonl'
done = {}
if os.path.exists(CACHE):
    for line in open(CACHE, encoding='utf-8'):
        r = json.loads(line); done[r['url']] = r
print('Already cached:', len(done))

kept = sum(1 for r in done.values() if r.get('ok'))
with open(CACHE, 'a', encoding='utf-8') as f:
    for url in tqdm(candidates):
        if kept >= VOA_MAX:
            break
        if url in done:
            continue
        try:
            art = parse_article(get(url))
            ok = bool(art['date']) and art['date'] < CUTOFF_DATE
            rec = {'url': url, 'ok': ok, **art}
        except Exception as e:
            rec = {'url': url, 'ok': False, 'error': str(e)}
        f.write(json.dumps(rec, ensure_ascii=False) + '\n'); f.flush()
        done[url] = rec
        kept += rec['ok']
        time.sleep(SLEEP_SEC)

voa_rows = []
for r in done.values():
    if r.get('ok'):
        p = make_passage(r['text'])
        if is_good(p):
            voa_rows.append({'text': p, 'title': r['title'], 'source': 'voa_khmer',
                             'url': r['url'], 'date': r['date'], 'license': 'US public domain (VOA)'})
voa_df = pd.DataFrame(voa_rows)
voa_df.to_csv(f'{DATA}/raw_voa.csv', index=False)
print(len(voa_df), 'VOA passages kept')
voa_df.head(3)

Already cached: 0


0it [00:00, ?it/s]

0 VOA passages kept


""


## Combine, de-duplicate, save `human.csv`

In [10]:
parts = [d for d in [wiki_df, globals().get('web_df'), globals().get('voa_df')] if d is not None and len(d)]
human = pd.concat(parts, ignore_index=True)
human['text'] = human['text'].map(normalize)
human = human[human['text'].map(is_good)]
human = human.drop_duplicates('text')
# near-duplicate check: same first 150 characters
human = human[~human['text'].str[:150].duplicated()]
human['id'] = human['text'].map(text_id)
human['label'] = 0
human['generator'] = 'human'
human = human[['id', 'text', 'title', 'source', 'url', 'date', 'license', 'generator', 'label']]
human.to_csv(f'{DATA}/human.csv', index=False)

print('TOTAL human passages:', len(human))
print(human['source'].value_counts())
print('\nLength (chars):'); print(human['text'].str.len().describe().round(0))

TOTAL human passages: 7964
source
mc4_web      3995
wikipedia    3969
Name: count, dtype: int64

Length (chars):
count    7964.0
mean     1484.0
std       715.0
min       300.0
25%       832.0
50%      1496.0
75%      2189.0
max      2500.0
Name: text, dtype: float64


## Done ✅
Send Claude the output of the last cell (counts + length stats). Next: **Notebook 2 — generate AI-written Khmer text** with free APIs (Gemini + Groq/OpenRouter).

*Limitation to note in your report:* the Wikipedia dump is from Nov 2023, so a few articles could have been edited with AI help; mC4 web text was crawled in 2013–2020 and VOA articles are filtered to before Nov 2022. mC4 pages keep the copyright of their original sites; use them for research only.